# DORA RAG — Phase 5b: Multi-Regulation Generation Layer

Extends Phase 5 to support DORA, NIS2, and PS21/3 with:
- Single-regulation `ask()` (DORA, NIS2, or PS21/3)
- Side-by-side comparison `ask_comparison()` — runs all three and shows each answer
- Conflict/gap detection `ask_with_analysis()` — flags overlaps, conflicts, and gaps

**All existing Phase 5 cells are preserved.** New cells are added after Cell 7.

**Prerequisites:** Phases 1–4b complete.
Expects: `dora_rag/index_config.json`, `dora_rag/retrieval_config.json`, `dora_rag/dora_chunks/all_chunks.json`

In [ ]:
# API NOTE
# The Gemini API key configured below is a paid-tier key with a fixed budget cap.
# It is intentionally set to a spending limit sufficient to run this pipeline.
# By running this code, you are authorised to make API calls under that budget.
# Do not share this key publicly.

In [ ]:
# Cell 1 — Install dependencies
import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'llama-index',
    'llama-index-llms-groq',
    'llama-index-llms-google-genai',
    'llama-index-embeddings-huggingface',
    'llama-index-vector-stores-chroma',
    'llama-index-retrievers-bm25',
    'chromadb',
    'sentence-transformers',
    'torch',
    'rank-bm25',
    'PyStemmer',
    'google-generativeai',
])
print('Dependencies installed.')


I0420 14:01:07.504815 10289343 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(95, generation: 1)
I0420 14:01:07.504877 10289343 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(79, generation: 1)


Dependencies installed.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


In [ ]:
import os
# Cell 2 — Imports, config, LLM setup
#
# LLM strategy:
#   Generation (ask/ask_single/ask_comparison) : Gemini 3 Flash
#     → 1M token context window — no token limit issues
#     → Free tier: 15 RPM, 1M TPM, 1500 req/day
#   Settings.llm (router internals)            : Groq 8b-instant
#     → Never used for actual generation

import json, time, datetime, re, torch
from pathlib import Path
from collections import Counter
from llama_index.core import Settings
from llama_index.llms.groq import Groq
from llama_index.llms.google_genai import GoogleGenAI

BASE_DIR    = Path('./dora_rag')
CHUNKS_DIR  = BASE_DIR / 'dora_chunks'
CHUNKS_FILE = CHUNKS_DIR / 'all_chunks.json'

# ── API keys ──────────────────────────────────────────────────────────────────
GROQ_API_KEY   = os.environ.get('GROQ_API_KEY', '')  # set in your environment or a .env file
GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY', '')  # set in your environment or a .env file

GROQ_ROUTER_MODEL = 'llama-3.1-8b-instant'
GEMINI_MODEL      = 'gemini-flash-latest'
RETRIEVAL_TOP_K   = 8  

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Router LLM: Groq 8b
router_llm   = Groq(model=GROQ_ROUTER_MODEL, api_key=GROQ_API_KEY)
Settings.llm = router_llm
print(f'✓ Settings.llm: Groq {GROQ_ROUTER_MODEL} (router only)')

# Generation LLM: Gemini 3 Flash
llm = GoogleGenAI(model=GEMINI_MODEL, api_key=GEMINI_API_KEY)
print(f'✓ Generation LLM: {GEMINI_MODEL}')
print(f'  Context window : 1M tokens — no chunk limit issues')

index_config     = json.loads((BASE_DIR / 'index_config.json').read_text())
retrieval_config = json.loads((BASE_DIR / 'retrieval_config.json').read_text())

CHROMA_DIR        = Path(index_config['chroma_dir'])
CHROMA_COLLECTION = index_config['chroma_collection']
EMBED_MODEL_NAME  = index_config['embed_model']

print(f'✓ Index  : {CHROMA_COLLECTION} ({index_config["total_vectors"]} vectors)')
print(f'✓ Corpus : {index_config.get("corpus_regulations", {})}')
print(f'✓ Device : {device}')
print(f'✓ Top-k  : {RETRIEVAL_TOP_K}')


✓ Settings.llm: Groq llama-3.1-8b-instant (router only)
✓ Generation LLM: models/gemini-flash-latest
  Context window : 1M tokens — no chunk limit issues
✓ Index  : dora_regulation (798 vectors)
✓ Corpus : {'dora': 381, 'nis2': 179, 'ps21_3': 119, 'cross_reg': 119}
✓ Device : cpu
✓ Top-k  : 8


/var/folders/lx/l412xvnx4r51cy8b4wx8rtjw0000gn/T/ipykernel_57770/184123701.py:38: DeprecationWarning: Call to deprecated class Gemini. (Should use `llama-index-llms-google-genai` instead, using Google's latest unified SDK. See: https://docs.llamaindex.ai/en/stable/examples/llm/google_genai/This package will no longer be supported after version 0.6.2) -- Deprecated since version 0.6.2.
  llm = Gemini(model=GEMINI_MODEL, api_key=GEMINI_API_KEY)


In [45]:
# Cell 3 — Load embedding model and Chroma index
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.vector_stores.chroma import ChromaVectorStore
import chromadb

print(f"Loading embedding model: {EMBED_MODEL_NAME}")
t0 = time.time()
embed_model = HuggingFaceEmbedding(
    model_name=EMBED_MODEL_NAME,
    query_instruction="Represent this sentence for searching relevant passages: ",
    embed_batch_size=16,
    device=device,
)
Settings.embed_model = embed_model
print(f"✓ Embedding model loaded in {time.time()-t0:.1f}s")

chroma_client     = chromadb.PersistentClient(path=str(CHROMA_DIR))
chroma_collection = chroma_client.get_collection(CHROMA_COLLECTION)
vector_store      = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context   = StorageContext.from_defaults(vector_store=vector_store)
index             = VectorStoreIndex.from_vector_store(
    vector_store=vector_store,
    embed_model=embed_model,
)
print(f"✓ Index loaded ({chroma_collection.count()} vectors)")

Loading embedding model: BAAI/bge-large-en-v1.5
✓ Embedding model loaded in 1.7s
✓ Index loaded (798 vectors)


In [46]:
# Cell 4 — Rebuild multi-regulation retrieval pipeline
from llama_index.core.schema import TextNode
from llama_index.core.retrievers import VectorIndexRetriever, QueryFusionRetriever
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.vector_stores import MetadataFilter, MetadataFilters, FilterOperator

raw_chunks = json.loads(CHUNKS_FILE.read_text(encoding='utf-8'))
all_nodes  = [
    TextNode(
        id_  = c['chunk_id'],
        text = c['text_with_header'],
        metadata = {
            'regulation':    c.get('regulation', 'dora'),
            'jurisdiction':  c.get('jurisdiction', 'eu'),
            'source_doc':    c['source_doc'],
            'doc_type':      c['doc_type'],
            'level':         c.get('level', 0),
            'article_num':   c['article_num'] if c['article_num'] is not None else -1,
            'article_title': c['article_title'],
            'section_ref':   c.get('section_ref') or '',
            'paragraph_idx': c['paragraph_idx'],
            'topic_tags':    '|'.join(c['topic_tags']),
            'entity_scope':  '|'.join(c['entity_scope']),
            'token_estimate':c['token_estimate'],
        }
    ) for c in raw_chunks
]

try:
    import Stemmer
    stemmer = Stemmer.Stemmer('english')
except ImportError:
    stemmer = None

def make_retriever(level=None, doc_type=None, regulation=None, jurisdiction=None, top_k=8):
    filters = []
    if level        is not None: filters.append(MetadataFilter(key='level',        value=level,        operator=FilterOperator.EQ))
    if doc_type     is not None: filters.append(MetadataFilter(key='doc_type',     value=doc_type,     operator=FilterOperator.EQ))
    if regulation   is not None: filters.append(MetadataFilter(key='regulation',   value=regulation,   operator=FilterOperator.EQ))
    if jurisdiction is not None: filters.append(MetadataFilter(key='jurisdiction', value=jurisdiction, operator=FilterOperator.EQ))
    if filters:
        return VectorIndexRetriever(index=index, similarity_top_k=top_k,
                                    filters=MetadataFilters(filters=filters))
    sem = VectorIndexRetriever(index=index, similarity_top_k=top_k)
    bm  = BM25Retriever.from_defaults(nodes=all_nodes, similarity_top_k=top_k,
                                       stemmer=stemmer, language='english')
    return QueryFusionRetriever(
        retrievers=[sem, bm], llm=Settings.llm, similarity_top_k=top_k,
        num_queries=1, mode='reciprocal_rerank', use_async=False, verbose=False,
    )

def detect_regulation(query):
    q, detected = query.lower(), []
    if any(k in q for k in ['dora','2022/2554','digital operational resilience','financial entit','eba','esma','eiopa']): detected.append('dora')
    if any(k in q for k in ['nis2','nis 2','2022/2555','network and information security','essential entit','important entit','enisa']): detected.append('nis2')
    if any(k in q for k in ['ps21','ps21/3','ss1/21','fg22','fca','pra','impact tolerance','important business service','self-assessment']): detected.append('ps21_3')
    return detected

def route_query(query, regulation=None):
    q = query.lower()
    if regulation:
        reg_scope = regulation
    else:
        detected  = detect_regulation(query)
        reg_scope = detected[0] if len(detected) == 1 else None
    if reg_scope == 'nis2':
        if any(k in q for k in ['implementing regulation','technical requirement','incident notification','2024/2690']):
            return make_retriever(regulation='nis2', level=2, top_k=10), 'NIS2 Level 2'
        if any(k in q for k in ['enisa','guideline','security measure']):
            return make_retriever(regulation='nis2', level=3, top_k=8), 'NIS2 Level 3'
        return make_retriever(regulation='nis2', top_k=10), 'NIS2 full corpus'
    if reg_scope == 'ps21_3':
        if any(k in q for k in ['ss1/21','supervisory statement','pra rule']): return make_retriever(regulation='ps21_3', doc_type='supervisory_statement', top_k=8), 'PS21/3 SS'
        if any(k in q for k in ['fg22','finalised guidance']): return make_retriever(regulation='ps21_3', doc_type='guidance', top_k=8), 'PS21/3 FCA Guidance'
        if any(k in q for k in ['critical third part','ctp','dp3/22']): return make_retriever(regulation='ps21_3', doc_type='discussion_paper', top_k=8), 'PS21/3 CTP'
        return make_retriever(regulation='ps21_3', top_k=10), 'PS21/3 full corpus'
    reg_filter = 'dora' if reg_scope == 'dora' else None
    if any(k in q for k in ['regulatory technical standard',' rts ','rts on','implementing technical standard',' its ','its on','technical standard','delegated regulation']):
        return make_retriever(level=2, regulation=reg_filter, top_k=12), 'Level 2 (RTS/ITS)'
    if any(k in q for k in ['tlpt','penetration test','threat-led','tiber']): return make_retriever(regulation=reg_filter, top_k=12), 'Full corpus (TLPT)'
    if any(k in q for k in ['subcontract','chain','group entit','concentrat','third-party provider']): return make_retriever(regulation=reg_filter, top_k=12), 'Full corpus (adversarial)'
    if any(k in q for k in ['guideline','aggregate','oversight cooperation','public statement']): return make_retriever(level=3, regulation=reg_filter, top_k=8), 'Level 3 (guidelines)'
    if any(k in q for k in ['template','reporting form']): return make_retriever(doc_type='ITS', regulation=reg_filter, top_k=8), 'ITS only'
    if any(k in q for k in ['article ','regulation ','directive ',' shall ',' must ','2022/2554']): return make_retriever(level=1, regulation=reg_filter, top_k=8), 'Level 1 (main regulation)'
    if reg_filter: return make_retriever(regulation=reg_filter, top_k=8), f'{reg_filter.upper()} full corpus'
    return make_retriever(top_k=10), 'Full corpus (all regulations)'

def retrieve(query, regulation=None, top_k=None, verbose=False):
    retriever, scope = route_query(query, regulation=regulation)
    results = retriever.retrieve(query)
    return results, scope

by_reg = Counter(c.get('regulation','?') for c in raw_chunks)
print(f'✓ Retrieval pipeline ready ({len(all_nodes)} chunks)')
for reg, cnt in sorted(by_reg.items()):
    print(f'  {reg:<12}: {cnt} chunks')


✓ Retrieval pipeline ready (826 chunks)
  cross_reg   : 128 chunks
  dora        : 390 chunks
  nis2        : 189 chunks
  ps21_3      : 119 chunks


In [47]:
# Cell 5 — DORA system prompt
#
# Carefully engineered for compliance-grade legal RAG:
# - Grounds every answer in retrieved context only
# - Enforces Article-level citation
# - Distinguishes SHALL / SHOULD / MAY obligations
# - Flags entity-specific applicability
# - Refuses to hallucinate when context is insufficient

from llama_index.core import PromptTemplate

DORA_QA_PROMPT = PromptTemplate("""\
You are a regulatory compliance assistant specialising in EU DORA \
(Digital Operational Resilience Act — Regulation (EU) 2022/2554) \
and its Level 2 regulatory technical standards and Level 3 guidelines.

CONTEXT (retrieved regulatory provisions):
---------------------------------------------
{context_str}
---------------------------------------------

QUESTION: {query_str}

INSTRUCTIONS — follow these rules exactly:
1. Base your answer ONLY on the context provided above. Do not use outside knowledge.
2. For every statement you make, cite the source using this format:
   [Source: <document_name>, Article <N>] or [Source: <document_name>] for non-article docs.
3. Clearly distinguish between:
   - SHALL / MUST  → mandatory obligation
   - SHOULD        → recommended but not mandatory
   - MAY           → optional / discretionary
4. If a provision applies only to specific entity types (e.g. credit institutions,
   investment firms, crypto-asset service providers), state this explicitly.
5. If the answer cannot be found in the provided context, respond with:
   "The provided context does not contain sufficient information to answer this question."
   Do NOT speculate or draw on outside knowledge.
6. Structure your answer clearly:
   - Start with a direct 1-2 sentence answer
   - Then provide detail with citations
   - End with a 'Key obligations summary' if the question involves multiple obligations

ANSWER:\
""")

print("✓ DORA system prompt defined")
print(f"  Prompt length: {len(DORA_QA_PROMPT.template)} chars")

✓ DORA system prompt defined
  Prompt length: 1425 chars


In [48]:
# Cell 6 — Build the query engine
#
# RetrieverQueryEngine wires together:
# retriever → context assembly → prompt → LLM → response

from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.response_synthesizers import get_response_synthesizer

# Response synthesizer: compact mode assembles all retrieved chunks
# into a single context window before calling the LLM once.
response_synthesizer = get_response_synthesizer(
    llm=llm,
    text_qa_template=DORA_QA_PROMPT,
    response_mode="compact",     # single LLM call with all context
    verbose=False,
)

hybrid_retriever = make_retriever()  # semantic + BM25, RRF

query_engine = RetrieverQueryEngine(
    retriever=hybrid_retriever,
    response_synthesizer=response_synthesizer,
)

print("✓ Query engine ready")
print("  Retriever : hybrid (semantic + BM25, RRF)")
print("  Synthesizer: compact (single LLM call)")
print("  LLM       : Groq / llama-3.3-70b-versatile")

✓ Query engine ready
  Retriever : hybrid (semantic + BM25, RRF)
  Synthesizer: compact (single LLM call)
  LLM       : Groq / llama-3.3-70b-versatile


In [49]:
# Cell 7 — Core ask() function (Gemini generation)
from llama_index.core import PromptTemplate

def llm_complete_with_retry(prompt, max_retries=4):
    """Call Gemini 2.5 Flash with automatic retry on rate limit errors."""
    for attempt in range(1, max_retries + 1):
        try:
            return llm.complete(prompt).text.strip()
        except Exception as e:
            err = str(e)
            is_rate = any(k in err for k in [
                'RESOURCE_EXHAUSTED', '429', 'quota', 'rate_limit',
                'retry_delay', 'Too Many Requests'
            ])
            if not is_rate or attempt == max_retries:
                raise
            match = re.search(r'retry_delay.*?(\d+)', err)
            wait  = int(match.group(1)) + 2 if match else 30
            print(f'    Gemini rate limit ({attempt}/{max_retries}). Waiting {wait}s...')
            time.sleep(wait)


def ask(question, verbose=True):
    """
    Ask a DORA compliance question (backwards-compatible with Phase 5).
    Uses Gemini 2.5 Flash for generation — no token limit issues.
    """
    t0 = time.time()
    nodes, scope = retrieve(question)

    context_parts = []
    for node in nodes:
        m      = node.metadata
        src    = m.get('source_doc', 'Unknown')
        art    = m.get('article_num', -1)
        title  = m.get('article_title', '')
        header = f'[{src}, Article {art} — {title}]' if art != -1 else f'[{src}]'
        context_parts.append(f'{header}\n{node.text}')

    context_str = '\n\n'.join(context_parts)
    prompt  = DORA_QA_PROMPT.format(context_str=context_str, query_str=question)
    answer  = llm_complete_with_retry(prompt)
    latency = round(time.time() - t0, 2)

    sources = []
    for node in nodes:
        m = node.metadata
        sources.append({
            'source_doc':    m.get('source_doc', ''),
            'doc_type':      m.get('doc_type', ''),
            'level':         m.get('level', 0),
            'article_num':   m.get('article_num', -1),
            'article_title': m.get('article_title', ''),
            'score':         round(getattr(node, 'score', 0.0) or 0.0, 4),
        })

    result = {'question': question, 'answer': answer, 'sources': sources,
               'scope': scope, 'latency_s': latency}

    if verbose:
        _display(result)
    return result


def _display(result):
    print('\n' + '═' * 70)
    print(f'QUESTION: {result["question"]}')
    print(f'Scope   : {result["scope"]}  |  Latency: {result["latency_s"]}s')
    print('─' * 70)
    print(result['answer'])
    print('\n' + '─' * 70)
    print('SOURCES RETRIEVED:')
    for i, s in enumerate(result['sources'], 1):
        art = f'Art {s["article_num"]}' if s['article_num'] != -1 else 'n/a'
        print(f'  [{i}] L{s["level"]} {s["doc_type"]:<22} {art:<8} '
              f'{s["source_doc"][:40]}  score: {s["score"]}')
    print('═' * 70)


print('✓ ask() function ready (Gemini 2.5 Flash)')
print('  Usage: result = ask("your question here")')


✓ ask() function ready (Gemini 2.5 Flash)
  Usage: result = ask("your question here")


In [50]:
# Cell 7b — Multi-regulation prompts
#
# Three prompt templates:
#   MULTI_REG_PROMPT       — single-regulation (DORA, NIS2, or PS21/3)
#   COMPARISON_PROMPT      — side-by-side across all three
#   CONFLICT_ANALYSIS_PROMPT — detects overlaps, conflicts, and gaps

from llama_index.core import PromptTemplate

# ── Single-regulation prompt (works for any of the three) ─────────────────
MULTI_REG_PROMPT = PromptTemplate("""\
You are a regulatory compliance expert specialising in:
- EU DORA (Digital Operational Resilience Act 2022/2554)
- EU NIS2 (Network and Information Security Directive 2022/2555)
- UK PS21/3 (FCA/PRA Operational Resilience Policy Statement)

CONTEXT (retrieved provisions from {regulation_label}):
---------------------------------------------
{context_str}
---------------------------------------------

QUESTION: {query_str}

INSTRUCTIONS:
1. Answer ONLY from the context above. Never use outside knowledge.
2. Cite every statement: [Source: <document>, Article <N>] for EU docs.
   For UK PS21/3 docs use: [Source: <document>, Section <ref>].
3. Distinguish: SHALL/MUST = mandatory, SHOULD = recommended, MAY = optional.
4. Flag entity-specific scope where relevant.
5. STRICT REFUSAL: If the primary topic is outside this regulation's scope,
   respond ONLY with: 'INSUFFICIENT CONTEXT: This topic falls outside the
   scope of {regulation_label}.'
6. Structure: direct answer first, then detail with citations.

ANSWER:\
""")

# ── Comparison prompt — runs after getting all three single answers ────────
COMPARISON_PROMPT = PromptTemplate("""\
You are a multi-jurisdiction regulatory compliance expert covering DORA, NIS2, and PS21/3.
You have been given three regulatory answers to the same question.

DORA ANSWER:
{dora_answer}

NIS2 ANSWER:
{nis2_answer}

PS21/3 ANSWER:
{ps213_answer}

QUESTION: {query_str}

Write a CROSS-FRAMEWORK ANALYSIS structured exactly as follows:

**OVERLAP:** [Where all three frameworks align or require the same thing]

**CONFLICT:** [Where frameworks differ — be specific about what each requires
and why this creates a compliance challenge for dual-regulated firms]

**GAP:** [Where one framework covers something others do not address]

**JURISDICTION NOTE:** [Which regulation applies to which type of firm,
and which firms face dual/triple compliance obligations]

Be specific. Cite articles and section references. If there is no meaningful
conflict or gap for a section, say 'None identified'.

ANALYSIS:\
""")

REG_LABELS = {
    'dora':   'EU DORA (Regulation 2022/2554)',
    'nis2':   'EU NIS2 (Directive 2022/2555)',
    'ps21_3': 'UK PS21/3 (FCA/PRA Operational Resilience)',
}

print('✓ Multi-regulation prompts defined')
print('  MULTI_REG_PROMPT   — single regulation answer')
print('  COMPARISON_PROMPT  — cross-framework analysis')
print('  All prompts use Gemini 2.5 Flash — full context, no truncation needed')


✓ Multi-regulation prompts defined
  MULTI_REG_PROMPT   — single regulation answer
  COMPARISON_PROMPT  — cross-framework analysis
  All prompts use Gemini 2.5 Flash — full context, no truncation needed


In [ ]:
# Cell 7c — Multi-regulation generation functions (Gemini)

def _build_context(nodes):
    """Build context string with correct citation format per jurisdiction."""
    parts = []
    for node in nodes:
        m   = node.metadata
        src = m.get('source_doc', 'Unknown')
        art = m.get('article_num', -1)
        sec = m.get('section_ref', '')
        ttl = m.get('article_title', '')
        if art != -1:
            hdr = f'[{src}, Article {art} — {ttl}]'
        elif sec:
            hdr = f'[{src}, Section {sec} — {ttl}]'
        else:
            hdr = f'[{src}]'
        parts.append(f'{hdr}\n{node.text}')
    return '\n\n'.join(parts)


def ask_single(question, regulation='dora', verbose=True):
    """
    Ask a question scoped to one regulation.
    regulation: 'dora' | 'nis2' | 'ps21_3'
    Uses Gemini 2.5 Flash — full context window, no truncation.
    """
    t0    = time.time()
    nodes, scope = retrieve(question, regulation=regulation)
    context_str  = _build_context(nodes)
    reg_label    = REG_LABELS.get(regulation, regulation)

    prompt = MULTI_REG_PROMPT.format(
        regulation_label=reg_label,
        context_str=context_str,
        query_str=question
    )
    answer  = llm_complete_with_retry(prompt)
    latency = round(time.time() - t0, 2)

    result = {'question': question, 'answer': answer, 'regulation': regulation,
               'scope': scope, 'latency_s': latency, 'nodes': nodes}

    if verbose:
        print(f'\n{"═"*70}')
        print(f'[{reg_label}]')
        print(f'Q: {question}')
        print(f'Scope: {scope} | {latency}s')
        print('─' * 70)
        print(answer)
        print('═' * 70)
    return result


def ask_comparison(question, verbose=True):
    """
    Side-by-side comparison across DORA, NIS2, and PS21/3.
    4 Gemini calls: 3 single-regulation answers + 1 comparison synthesis.
    Gemini free tier: 15 RPM — brief pauses keep well within limit.
    """
    t0 = time.time()
    if verbose:
        print(f'\n{"═"*70}')
        print(f'COMPARISON QUERY: {question}')
        print(f'{"═"*70}')

    answers = {}
    for reg in ['dora', 'nis2', 'ps21_3']:
        if verbose: print(f'  [{REG_LABELS[reg]}] retrieving...')
        r = ask_single(question, regulation=reg, verbose=False)
        answers[reg] = r
        time.sleep(4)   # stay comfortably within Gemini 15 RPM

    if verbose: print('  Generating cross-framework analysis...')
    comparison_prompt = COMPARISON_PROMPT.format(
        dora_answer  = answers['dora']['answer'],
        nis2_answer  = answers['nis2']['answer'],
        ps213_answer = answers['ps21_3']['answer'],
        query_str    = question
    )
    analysis      = llm_complete_with_retry(comparison_prompt)
    total_latency = round(time.time() - t0, 2)

    result = {'question': question, 'answers': answers,
               'analysis': analysis, 'latency_s': total_latency}

    if verbose:
        print(f'\n{"─"*70}\n[EU DORA]')
        print(answers['dora']['answer'])
        print(f'\n{"─"*70}\n[EU NIS2]')
        print(answers['nis2']['answer'])
        print(f'\n{"─"*70}\n[UK PS21/3]')
        print(answers['ps21_3']['answer'])
        print(f'\n{"─"*70}\nCROSS-FRAMEWORK ANALYSIS')
        print('─' * 70)
        print(analysis)
        print(f'\nTotal latency: {total_latency}s (4 Gemini calls)')
        print('═' * 70)
    return result


print('✓ Multi-regulation generation functions ready (Gemini 3 Flash)')
print('  ask_single(question, regulation="dora")   — one regulation')
print('  ask_comparison(question)                  — all three + analysis')
print('  ask(question)                             — DORA only (Phase 5 compatible)')


✓ Multi-regulation generation functions ready (Gemini 2.5 Flash)
  ask_single(question, regulation="dora")   — one regulation
  ask_comparison(question)                  — all three + analysis
  ask(question)                             — DORA only (Phase 5 compatible)


In [52]:
# Cell 7d — Test ask_single() on each regulation
# Uses 3 LLM calls — one per regulation
INCIDENT_Q = 'What are the incident reporting obligations and timelines?'

dora_result  = ask_single(INCIDENT_Q, regulation='dora')
time.sleep(3)
nis2_result  = ask_single(INCIDENT_Q, regulation='nis2')
time.sleep(3)
ps213_result = ask_single(INCIDENT_Q, regulation='ps21_3')



══════════════════════════════════════════════════════════════════════
[EU DORA (Regulation 2022/2554)]
Q: What are the incident reporting obligations and timelines?
Scope: DORA full corpus | 58.3s
──────────────────────────────────────────────────────────────────────
Financial entities are required to report major ICT-related incidents through a structured process involving an initial notification, an intermediate report, and a final report. These reports must be submitted using specific templates and adhere to classification criteria and notification rules regarding outsourcing.

### 1. Incident Classification and Scope
*   **Major Incident Criteria:** An incident is classified as "major" if it affects critical services and meets the materiality threshold in Article 9(5)(b), or meets two or more other materiality thresholds [Source: L2_RTS_ICT_Incident_Classification, Article 8(1)].
*   **Recurring Incidents:** Incidents that are not individually major SHALL be considered a single m

In [53]:
# Cell 7e — Test ask_comparison() on a cross-regulation question
# Uses 4 LLM calls — run when you have quota available
# The most important new capability in Phase 5b

comparison_result = ask_comparison(
    'What are the incident reporting requirements and timelines?'
)



══════════════════════════════════════════════════════════════════════
COMPARISON QUERY: What are the incident reporting requirements and timelines?
══════════════════════════════════════════════════════════════════════
  [EU DORA (Regulation 2022/2554)] retrieving...
  [EU NIS2 (Directive 2022/2555)] retrieving...
  [UK PS21/3 (FCA/PRA Operational Resilience)] retrieving...
  Generating cross-framework analysis...

──────────────────────────────────────────────────────────────────────
[EU DORA]
Under EU DORA (Regulation 2022/2554) and its associated technical standards, financial entities are required to identify, classify, and report major ICT-related incidents through a three-stage process consisting of an initial notification, an intermediate report, and a final report.

### 1. Incident Classification and Scope
*   **Major Incident Criteria:** An incident SHALL be considered "major" if it affects critical services and meets the materiality threshold in Article 9(5)(b) or meets two

In [54]:
# Cell 7f — Test hallucination guard across regulations
# Each question is out-of-scope for the regulation it's asked to

# GDPR question asked of DORA
ask_single('What are the GDPR data subject access request obligations?', regulation='dora')
time.sleep(3)

# PS21/3 question asked of NIS2
ask_single('What are the impact tolerance requirements for important business services?', regulation='nis2')



══════════════════════════════════════════════════════════════════════
[EU DORA (Regulation 2022/2554)]
Q: What are the GDPR data subject access request obligations?
Scope: DORA full corpus | 3.72s
──────────────────────────────────────────────────────────────────────
INSUFFICIENT CONTEXT: This topic falls outside the scope of EU DORA (Regulation 2022/2554).
══════════════════════════════════════════════════════════════════════

══════════════════════════════════════════════════════════════════════
[EU NIS2 (Directive 2022/2555)]
Q: What are the impact tolerance requirements for important business services?
Scope: NIS2 full corpus | 15.41s
──────────────────────────────────────────────────────────────────────
INSUFFICIENT CONTEXT: This topic falls outside the scope of EU NIS2 (Directive 2022/2555).
══════════════════════════════════════════════════════════════════════


{'question': 'What are the impact tolerance requirements for important business services?',
 'answer': 'INSUFFICIENT CONTEXT: This topic falls outside the scope of EU NIS2 (Directive 2022/2555).',
 'regulation': 'nis2',
 'scope': 'NIS2 full corpus',
 'latency_s': 15.41,
 'nodes': [NodeWithScore(node=TextNode(id_='NIS2_IR_2024_2690_Cy__art014__p00', embedding=None, metadata={'regulation': 'nis2', 'jurisdiction': 'eu', 'source_doc': 'NIS2_IR_2024_2690_Cybersecurity_Requirements', 'doc_type': 'implementing_regulation', 'level': 2, 'article_num': 14, 'article_title': 'Significant incidents with regard to trust service providers', 'section_ref': '', 'paragraph_idx': 0, 'topic_tags': 'incident_reporting|third_party_risk|data_security|network_security', 'entity_scope': '', 'token_estimate': 297}, excluded_embed_metadata_keys=['paragraph_idx', 'token_estimate'], excluded_llm_metadata_keys=['token_estimate'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text='A

In [55]:
# Cell 8 — First end-to-end test
# A straightforward factual lookup — good first sanity check

result = ask(
    "What are the timelines for reporting a major ICT-related incident "
    "to the competent authority under DORA?"
)


══════════════════════════════════════════════════════════════════════
QUESTION: What are the timelines for reporting a major ICT-related incident to the competent authority under DORA?
Scope   : DORA full corpus  |  Latency: 189.9s
──────────────────────────────────────────────────────────────────────
The provided context does not contain sufficient information to answer this question regarding the specific numerical time limits (such as the exact number of hours or days) for submitting the initial notification to the competent authority, as these are to be laid down in accordance with Article 20, which is not included in the provided text.

However, based on the provided context, the following relative timelines and reporting stages apply to major ICT-related incidents:

### Reporting Stages and Relative Timelines
Financial entities **shall** submit the following to the relevant competent authority:
*   **Initial Notification:** Required, but specific time limits are not provided in

In [ ]:
# Cell 9a. Test across all DORA query types

TEST_QUESTIONS = [
    # Factual — specific obligation
    "What must a financial entity include in the initial notification of a major ICT incident?",

    # Obligation type — shall vs may
    "Is it mandatory for financial entities to appoint an ICT third-party risk function?",

    # Applicability
    "Does DORA apply to crypto-asset service providers, and if so what are their obligations?",

    # Cross-level — Level 1 + Level 2
    "What does the main regulation say about TLPT, and how is this further specified in the RTS?",

    # Cross-reference
    "What contractual provisions must be included when outsourcing ICT services under Article 30?",

    # Simplified framework
    "What is the simplified ICT risk management framework and which entities can use it?",

    # Negative test — DORA does not cover this
    "What does DORA say about GDPR data subject rights?",
]

results = []
for q in TEST_QUESTIONS:
    r = ask(q)
    results.append(r)
    print()  # spacing between answers


══════════════════════════════════════════════════════════════════════
QUESTION: What must a financial entity include in the initial notification of a major ICT incident?
Scope   : Level 1 (main regulation)  |  Latency: 49.9s
──────────────────────────────────────────────────────────────────────
A financial entity must include all information necessary for the competent authority to determine the significance of the major ICT-related incident and to assess potential cross-border impacts. This notification must be prepared after collecting and analyzing all relevant information and must be submitted using specific regulatory templates.

### Detailed Requirements for Initial Notification

According to the provided context, the requirements for the initial notification of a major ICT-related incident are as follows:

*   **Mandatory Content (SHALL):** The initial notification shall include all information necessary for the competent authority to determine the significance of the major IC

In [ ]:
#Cell 9b. Pinpoint where the hang is
import time

q = TEST_QUESTIONS[0]
print(f"Testing: {q[:50]}")

print("Step 1: route_query...")
t0 = time.time()
retriever, scope = route_query(q)
print(f"  ✓ {time.time()-t0:.1f}s — scope: {scope}")

print("Step 2: retrieve...")
t0 = time.time()
nodes = retriever.retrieve(q)
print(f"  ✓ {time.time()-t0:.1f}s — {len(nodes)} nodes")

print("Step 3: build context...")
t0 = time.time()
context_str = _build_context(nodes)
print(f"  ✓ {time.time()-t0:.1f}s — {len(context_str)} chars")

print("Step 4: Gemini call...")
t0 = time.time()
prompt = DORA_QA_PROMPT.format(context_str=context_str, query_str=q)
answer = llm_complete_with_retry(prompt)
print(f"  ✓ {time.time()-t0:.1f}s")
print(f"\nAnswer preview: {answer[:200]}")

Testing: What must a financial entity include in the initia
Step 1: route_query...
  ✓ 0.0s — scope: Level 1 (main regulation)
Step 2: retrieve...
  ✓ 0.1s — 8 nodes
Step 3: build context...
  ✓ 0.0s — 22879 chars
Step 4: Gemini call...
  ✓ 8.5s

Answer preview: A financial entity must include all information necessary for the competent authority to determine the significance of the major ICT-related incident and assess possible cross-border impacts. This not


In [ ]:
# Cell 10. Response quality checks
#
# Automated checks on every answer:
# 1. Citation presence — does the answer contain [Source: ...] citations?
# 2. Obligation flagging — does it distinguish SHALL/SHOULD/MAY?
# 3. Hallucination guard — does the 'not in context' fallback work?
# 4. Answer length — not too short (incomplete) or too long (verbose)

def check_response_quality(result: dict) -> dict:
    answer = result["answer"]

    # Citation check
    has_citation    = bool(re.search(r'\[Source:', answer, re.IGNORECASE))

    # Obligation language check
    has_obligation  = bool(re.search(r'\b(shall|must|should|may)\b', answer, re.IGNORECASE))

    # Hallucination guard check (for negative test)
    has_not_found   = bool(re.search(
        r'does not contain|not found|cannot be found|insufficient information',
        answer, re.IGNORECASE
    ))

    word_count      = len(answer.split())
    length_ok       = 30 <= word_count <= 600

    return {
        "question":       result["question"][:55],
        "has_citation":   has_citation,
        "has_obligation": has_obligation,
        "has_not_found":  has_not_found,
        "word_count":     word_count,
        "length_ok":      length_ok,
        "latency_s":      result["latency_s"],
    }


print("RESPONSE QUALITY REPORT")
print("=" * 90)
print(f"  {'#':<3} {'Cite':>5} {'Oblig':>6} {'Len':>5} {'Lat':>5}  Question")
print("  " + "─" * 85)

all_results = []
for i, q in enumerate(TEST_QUESTIONS, 1):
    print(f"  [{i}/{len(TEST_QUESTIONS)}] {q[:55]}...")
    try:
        r = ask(q, verbose=False)
        all_results.append(r)
        print(f"    ✓ {r['latency_s']}s")
    except Exception as e:
        print(f"    ✗ Failed: {e}")
        # Still append a placeholder so downstream cells don't break
        all_results.append({
            'question': q, 'answer': '', 'sources': [],
            'scope': 'error', 'latency_s': 0
        })
    # Explicit pause between calls — prevents Gemini silent stall
    if i < len(TEST_QUESTIONS):
        time.sleep(5)

# Summary
cite_rate  = sum(1 for r in all_results if re.search(r'\[Source:', r["answer"], re.IGNORECASE))
oblig_rate = sum(1 for r in all_results if re.search(r'\b(shall|must|should|may)\b', r["answer"], re.IGNORECASE))
avg_lat    = round(sum(r["latency_s"] for r in all_results) / len(all_results), 2)

print("  " + "─" * 85)  
print(f"\n  Citation rate    : {cite_rate}/{len(all_results)}")
print(f"  Obligation rate  : {oblig_rate}/{len(all_results)}")
print(f"  Avg latency      : {avg_lat}s")

RESPONSE QUALITY REPORT
  #    Cite  Oblig   Len   Lat  Question
  ─────────────────────────────────────────────────────────────────────────────────────
  [1/7] What must a financial entity include in the initial not...
    ✓ 10.53s
  [2/7] Is it mandatory for financial entities to appoint an IC...
    ✓ 8.88s
  [3/7] Does DORA apply to crypto-asset service providers, and ...
    ✓ 10.96s
  [4/7] What does the main regulation say about TLPT, and how i...
    ✗ Failed: 504 Deadline expired before operation could complete.
  [5/7] What contractual provisions must be included when outso...
    ✓ 37.47s
  [6/7] What is the simplified ICT risk management framework an...
    ✓ 15.43s
  [7/7] What does DORA say about GDPR data subject rights?...
    ✓ 22.06s
  ─────────────────────────────────────────────────────────────────────────────────────

  Citation rate    : 5/7
  Obligation rate  : 5/7
  Avg latency      : 15.05s


In [59]:
# Cell 11 — Faithfulness & relevancy evaluation (direct Gemini judge)
# Bypasses LlamaIndex evaluators — no internal vector index, no async conflicts
# 2 Gemini calls per question: 1 generation + 1 judge

import json as _json

EVAL_QUESTIONS = [
    "What are the reporting timelines for a major ICT incident under DORA?",
    "What contractual provisions must be included when outsourcing ICT services?",
    "What does DORA say about GDPR data subject rights?",   # negative test
]

JUDGE_PROMPT = """\
You are evaluating a RAG system answer. Score it on two dimensions.

QUESTION: {question}

RETRIEVED CONTEXT:
{context}

ANSWER:
{answer}

Respond with ONLY this JSON (no markdown, no explanation):
{{"faithfulness": 1 or 0, "relevancy": 1 or 0, "reason": "one sentence"}}

faithfulness = 1 if every claim in the answer is supported by the context, else 0
relevancy    = 1 if the answer addresses the question, else 0
"""

print("Running faithfulness & relevancy evaluation (direct Gemini judge)...")
print("(2 Gemini calls per question — no LlamaIndex evaluator overhead)\n")

eval_results = []
for i, q in enumerate(EVAL_QUESTIONS, 1):
    print(f"  [{i}/{len(EVAL_QUESTIONS)}] {q[:55]}...")
    t0 = time.time()

    # Step 1: generate answer
    nodes, scope = retrieve(q)
    context_str  = _build_context(nodes)
    prompt       = DORA_QA_PROMPT.format(context_str=context_str, query_str=q)
    answer       = llm_complete_with_retry(prompt)

    # Step 2: judge answer
    judge_prompt = JUDGE_PROMPT.format(
        question=q,
        context=context_str[:3000],
        answer=answer
    )
    judge_raw = llm_complete_with_retry(judge_prompt)

    # Parse JSON
    try:
        clean  = re.sub(r'```json|```', '', judge_raw).strip()
        scores = _json.loads(clean)
        faithful = bool(scores.get('faithfulness', 0))
        relevant = bool(scores.get('relevancy', 0))
        reason   = scores.get('reason', '')
    except Exception:
        faithful, relevant, reason = False, False, f"parse error: {judge_raw[:80]}"

    latency = round(time.time() - t0, 2)
    eval_results.append({
        "question":  q[:55],
        "faithful":  faithful,
        "relevant":  relevant,
        "reason":    reason,
        "latency_s": latency,
    })

    print(f"     Faithful : {'✓' if faithful else '✗'}  "
          f"Relevant: {'✓' if relevant else '✗'}  ({latency}s)")
    print(f"     Reason   : {reason}")
    print()

    if i < len(EVAL_QUESTIONS):
        time.sleep(5)

faith_pass = sum(1 for r in eval_results if r["faithful"])
rel_pass   = sum(1 for r in eval_results if r["relevant"])
print(f"Faithfulness : {faith_pass}/{len(eval_results)} passing")
print(f"Relevancy    : {rel_pass}/{len(eval_results)} passing")

Running faithfulness & relevancy evaluation (direct Gemini judge)...
(2 Gemini calls per question — no LlamaIndex evaluator overhead)

  [1/3] What are the reporting timelines for a major ICT incide...
     Faithful : ✗  Relevant: ✓  (160.71s)
     Reason   : The answer cites and provides detailed information from Article 19 and Article 5, which are not present in the provided retrieved context.

  [2/3] What contractual provisions must be included when outso...
     Faithful : ✗  Relevant: ✓  (151.37s)
     Reason   : The answer includes extensive information from Article 28, Article 30(3)(b-f), and various recitals that are not present in the provided retrieved context.

  [3/3] What does DORA say about GDPR data subject rights?...
     Faithful : ✗  Relevant: ✓  (21.74s)
     Reason   : The answer includes information and citations from Article 6 and various L2 RTS documents that are not present in the provided context.

Faithfulness : 0/3 passing
Relevancy    : 3/3 passing


In [ ]:
# Cell 12 — Interactive query loop
# Type any DORA question and get a cited answer.
# Type 'quit' or 'exit' to stop.

print("DORA RAG — Interactive Query Mode")
print("=" * 50)
print("Type a DORA compliance question and press Enter.")
print("Type 'quit' to exit.\n")

while True:
    try:
        question = input("Question: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\nExiting.")
        break

    if not question:
        continue
    if question.lower() in ("quit", "exit", "q"):
        print("Exiting.")
        break

    ask(question)

DORA RAG — Interactive Query Mode
Type a DORA compliance question and press Enter.
Type 'quit' to exit.


══════════════════════════════════════════════════════════════════════
QUESTION: What does DORA require for threat-led penetration testing and how is this specified in the RTS?
Scope   : Full corpus (TLPT)  |  Latency: 25.39s
──────────────────────────────────────────────────────────────────────
DORA requires identified financial entities to conduct advanced digital operational resilience testing through Threat-Led Penetration Testing (TLPT) to simulate real-life cyber attacks on their critical production systems. This testing must be performed at least every three years and involves strict requirements for scope, tester qualifications, and the inclusion of third-party service providers.

### Detailed Requirements for TLPT

**1. Definition and Nature of TLPT**
TLPT is defined as a framework that mimics the tactics, techniques, and procedures (TTPs) of real-life threat actors. It mu

In [61]:
# Cell 13 — Save pipeline config for Phase 6 (evaluation)

pipeline_config = {
    "created_at":       datetime.datetime.now().isoformat(),
    "llm":              f"gemini/{GEMINI_MODEL}",
    "embed_model":      EMBED_MODEL_NAME,
    "retrieval_top_k":  RETRIEVAL_TOP_K,
    "retriever":        "hybrid (semantic + BM25, RRF)",
    "response_mode":    "compact",
    "prompt_version":   "v1",
    "corpus_levels":    index_config.get("corpus_levels", {}),
    "total_vectors":    index_config["total_vectors"],
    "quality_checks": {
        "citation_rate":    f"{cite_rate}/{len(all_results)}",
        "obligation_rate":  f"{oblig_rate}/{len(all_results)}",
        "avg_latency_s":    avg_lat,
        "faithfulness":     f"{faith_pass}/{len(eval_results)}",
        "relevancy":        f"{rel_pass}/{len(eval_results)}",
    },
    "notes": (
        "Baseline RAG pipeline. "
        "Phase 6 will add systematic evaluation across larger question sets "
        "and identify areas for improvement."
    )
}

pipeline_config_path = BASE_DIR / "pipeline_config.json"
pipeline_config_path.write_text(json.dumps(pipeline_config, indent=2))
print(f"✓ Pipeline config saved: {pipeline_config_path}")
print(json.dumps(pipeline_config, indent=2))

✓ Pipeline config saved: dora_rag/pipeline_config.json
{
  "created_at": "2026-04-20T17:37:24.316889",
  "llm": "gemini/models/gemini-flash-latest",
  "embed_model": "BAAI/bge-large-en-v1.5",
  "retrieval_top_k": 8,
  "retriever": "hybrid (semantic + BM25, RRF)",
  "response_mode": "compact",
  "prompt_version": "v1",
  "corpus_levels": {
    "1": 276,
    "2": 440,
    "3": 82
  },
  "total_vectors": 798,
  "quality_checks": {
    "citation_rate": "5/7",
    "obligation_rate": "5/7",
    "avg_latency_s": 15.05,
    "faithfulness": "0/3",
    "relevancy": "3/3"
  },
  "notes": "Baseline RAG pipeline. Phase 6 will add systematic evaluation across larger question sets and identify areas for improvement."
}


In [ ]:
#Cell 14 (Optional). Quick faithfulness check — are citations actually grounded in context?
q = "What are the timelines for reporting a major ICT incident under DORA?"
nodes, scope = retrieve(q)
context_str  = _build_context(nodes)
prompt       = DORA_QA_PROMPT.format(context_str=context_str, query_str=q)
answer       = llm_complete_with_retry(prompt)

print("ANSWER:")
print(answer[:800])
print()

# Manual check — do the cited articles appear in the retrieved context?
cited = re.findall(r'\[Source: ([^\]]+)\]', answer)
print(f"Citations found: {len(cited)}")
for c in cited:
    print(f"  {c}")

ANSWER:
Under DORA, the timelines for reporting major ICT-related incidents are structured into three stages for competent authorities and include immediate notification requirements for affected clients. While qualitative timelines are provided, the specific numerical deadlines (e.g., exact hours or days) for reporting to competent authorities are not contained in the provided context.

### 1. Reporting to Clients
*   **Major ICT-related incidents:** Financial entities **shall**, without undue delay as soon as they become aware of it, inform their clients if an incident has an impact on the clients' financial interests. They **must** also inform them of measures taken to mitigate adverse effects. [Source: L1_DORA_Regulation_2022_2554, Article 19(3)]
*   **Significant cyber threats:** Financial en

Citations found: 9
  L1_DORA_Regulation_2022_2554, Article 19(3)
  L1_DORA_Regulation_2022_2554, Article 19(3)
  L1_DORA_Regulation_2022_2554, Article 19(4)(a)
  L1_DORA_Regulation_2022_2554